In [1]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
INPUT_JSON = DATA_DIR / "mxy_wwsd.json"

with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 原始数据
df = pd.DataFrame(data)

In [2]:
# 初步展平
df_normalized = pd.json_normalize(data)

In [3]:
# 和初版数据进行对比，而不是直接跟清理后的对比
DATA = DATA_DIR / "list_with_detail.csv"
my_df = pd.read_csv(DATA)

In [4]:
# 调一下行的顺序，并按id升序排列
print(my_df.columns)
my_df = my_df[['id','name','description','size','museumName','imgUrl',
               'pictureIds','local_image_paths','image_count',
               'yearName','categoryName','collectionsCategory','collectionTexture',
               'collectedCounts', 'collectionLevel', 'threeUrl', 'fVideo','fAudio',
         'collectionUnit', 'clickCounts',  'isHighQuality']]
my_df = my_df.sort_values(['id'])

Index(['collectedCounts', 'collectionLevel', 'threeUrl', 'fAudio',
       'description', 'museumName', 'collectionUnit', 'pictureIds',
       'categoryName', 'fVideo', 'collectionsCategory', 'imgUrl', 'name',
       'clickCounts', 'id', 'isHighQuality', 'yearName', 'size',
       'collectionTexture', 'local_image_paths', 'image_count'],
      dtype='object')


In [5]:
# 观察两边的同一文物，发现mxy的数据只包含一张图，而我是对每件文物爬了多张图的
# 经过回顾发现我现在也无法重新爬取多张图了，这些图的url也找不回来了。下面考虑先对比一下mxy的图和我的是否相同
item1 = df[0:1].to_dict(orient='records')
my_item1 = my_df[my_df["id"]=="0D5EFF6837A34A73987DCADAF2CA50D3"].to_dict(orient='records')
# 对比后发现对于这一个文物而言确实是相同的

In [6]:
# 先看是不是df中的所有id都在my_df中，求一下二者的对称差
ids_in_df = set(df['id'])
ids_in_my_df = set(my_df['id'])
ids_in_df_not_in_my_df = ids_in_df - ids_in_my_df
ids_in_my_df_not_in_df = ids_in_my_df - ids_in_df
print(f"在df中但不在my_df中的id数量: {len(ids_in_df_not_in_my_df)}")
print(f"在my_df中但不在df中的id数量: {len(ids_in_my_df_not_in_df)}")

在df中但不在my_df中的id数量: 0
在my_df中但不在df中的id数量: 1764


In [7]:
# 确认了df中的id都在my_df中，看看这些id是否都只有一张图片，也就是观察其images列的元素数量
df_img_count = df['images'].apply(lambda x: len(x) if isinstance(x, list) else 0)
# 观察后发现，有8件文物没有图，其他都只有一张图。（说明mxy很可能没有爬取详情页，而是只是用了列表页的数据）

In [8]:
# 先检查一下df中的每个id对应的name是否与该id在my_df中的name一致，如果一致，说明至少在名称上两边的数据是匹配的
flag = True
for idx, row in df.iterrows():
    id = row['id']
    name_in_df = row['name']
    name_in_my_df = my_df[my_df['id'] == id]['name'].values[0]
    if name_in_df != name_in_my_df:
        print(f"ID {id} 的名称不匹配: df中的名称是 '{name_in_df}'，my_df中的名称是 '{name_in_my_df}'")
        flag = False
if flag:
    print("所有ID的名称在df和my_df中都匹配")

所有ID的名称在df和my_df中都匹配


In [9]:
# 可以发现所有名称都匹配，下面检查图片
# 具体而言，对于df中的每一行，获取其images列（这是一个对象数组，只需取其第一个元素，若没有元素则跳过）中的id字段和url字段，然后与my_df中对应id的imgUrl字段进行对比，看看是否相同
for index, row in df.iterrows():
    item_id = row['id']
    images = row['images']
    if isinstance(images, list) and len(images) > 0:
        first_image = images[0]
        # image_id = first_image.get('id')
        # # 发现mxy给一部分图片的id加了后缀，这些后缀多以"-"或"_"开头，后面跟着一些数字或字母。下面先去掉这些后缀再进行对比
        # if isinstance(image_id, str):
        #     image_id_splited = image_id.split('-')[0].split('_')[0]
        # if image_id_splited != item_id:
        #     print(f"文物ID {item_id} 的图片ID在去掉后缀后与文物ID不匹配，其原始图片ID为 {image_id}，")
        image_url = first_image.get('url')
        
        # 在my_df中找到对应id的行
        my_row = my_df[my_df['id'] == item_id]
        if not my_row.empty:
            my_image_url = my_row.iloc[0]['imgUrl']
            if image_url != my_image_url:
                print(f"文物ID {item_id} 的图片URL与我爬取的不匹配：mxy的url为 {image_url} ，但我的url为 {my_image_url}")

文物ID F02B9C5AF199485BBB332AEBB43E1E63 的图片URL与我爬取的不匹配：mxy的url为 http://www.wwsdw.net/sdimg/picture/shandong/37028121800001/00001525/thumb/640x426_3702812180000100001525-C-0001.jpg ，但我的url为 http://www.wwsdw.net/sdimg/picture/shandong/37040221800003/00006326/thumb/640x426_3704022180000300006326-A-0001.jpg
文物ID a7480a7d8fcb454eb58dceb038750ce1 的图片URL与我爬取的不匹配：mxy的url为 http://www.wwsdw.net/sdimg/picture/thumbs/a748/640x426_a7480a7d8fcb454eb58dceb038750ce1.JPG ，但我的url为 http://www.wwsdw.net/sdimg/picture/thumbs/dd64/640x426_dd64531f37254504b5de2390b6eb84fa.JPG
文物ID 29e37036624a48a687a3103e5b1f701a 的图片URL与我爬取的不匹配：mxy的url为  ，但我的url为 nan
文物ID 51c9806ebe6a4f7f98cfea3fcb64ccbb 的图片URL与我爬取的不匹配：mxy的url为  ，但我的url为 nan


In [10]:
# 可以发现有两件文物的url不匹配，其中文物a7480a7d8fcb454eb58dceb038750ce1，mxy爬取的url是404的，我的正确；对于文物F02B9C5AF199485BBB332AEBB43E1E63，mxy爬取的url实际是文物F0BCE7BF24AD45E991ED5635DD883D7C的图片，可能是出现了错误，见以下的代码及输出。
from pprint import pprint
pprint(df[df["id"]=="F02B9C5AF199485BBB332AEBB43E1E63"]["images"].values[0])
pprint(my_df[my_df["id"]=="F02B9C5AF199485BBB332AEBB43E1E63"]["imgUrl"].values[0])
pprint(my_df[my_df["imgUrl"]=="http://www.wwsdw.net/sdimg/picture/shandong/37028121800001/00001525/thumb/640x426_3702812180000100001525-C-0001.jpg"]["id"])
pprint(df[df["id"]=="F0BCE7BF24AD45E991ED5635DD883D7C"]["images"].values[0])

[{'description': None,
  'extension': 'jpg',
  'id': 'F02B9C5AF199485BBB332AEBB43E1E63_img_0',
  'name': None,
  'url': 'http://www.wwsdw.net/sdimg/picture/shandong/37028121800001/00001525/thumb/640x426_3702812180000100001525-C-0001.jpg'}]
'http://www.wwsdw.net/sdimg/picture/shandong/37040221800003/00006326/thumb/640x426_3704022180000300006326-A-0001.jpg'
3704    F0BCE7BF24AD45E991ED5635DD883D7C
Name: id, dtype: object
[{'description': None,
  'extension': 'jpg',
  'id': 'F0BCE7BF24AD45E991ED5635DD883D7C_img_0',
  'name': None,
  'url': 'http://www.wwsdw.net/sdimg/picture/shandong/37028121800001/00001525/thumb/640x426_3702812180000100001525-C-0001.jpg'}]


In [11]:
target_ids = ["F02B9C5AF199485BBB332AEBB43E1E63", "F0BCE7BF24AD45E991ED5635DD883D7C"]
filtered_df = df[df['id'].isin(target_ids)]
filtered_df

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,currentLocation,excavationDate,sourceCitation,collectionInfo,images,shape_type
1301,F02B9C5AF199485BBB332AEBB43E1E63,新石器时代大汶口文化黑衣红褐陶壶,新石器时代,大汶口文化,None,None,[],暂无描述,"[{'label': 'material', 'value': '黑衣红褐陶'}]","{'country': None, 'province': None, 'city': No...","{'country': None, 'province': None, 'city': No...",None,"{'sourceId': 'wwsd', 'locator': '', 'locatorTy...","{'collectorName': '', 'collectorId': None, 'co...",[{'id': 'F02B9C5AF199485BBB332AEBB43E1E63_img_...,壶
1302,F0BCE7BF24AD45E991ED5635DD883D7C,新石器时代大汶口文化带盖陶鼎,新石器时代,大汶口文化,None,None,[],暂无描述,"[{'label': 'material', 'value': '陶'}]","{'country': None, 'province': None, 'city': No...","{'country': None, 'province': None, 'city': No...",None,"{'sourceId': 'wwsd', 'locator': '', 'locatorTy...","{'collectorName': '', 'collectorId': None, 'co...",[{'id': 'F0BCE7BF24AD45E991ED5635DD883D7C_img_...,鼎


In [12]:
# 综上可知，mxy的数据大部分都是和我一致的，仅有两个图片url发生错误。
# 下面将我的数据逐步规约至和mxy相同的格式。逐步处理各列。
print(df.columns)
print(my_df.columns)


Index(['id', 'name', 'era', 'culture', 'time', 'dimensionsDesc',
       'structuredDimensions', 'fullDesc', 'features', 'excavationLocation',
       'currentLocation', 'excavationDate', 'sourceCitation', 'collectionInfo',
       'images', 'shape_type'],
      dtype='object')
Index(['id', 'name', 'description', 'size', 'museumName', 'imgUrl',
       'pictureIds', 'local_image_paths', 'image_count', 'yearName',
       'categoryName', 'collectionsCategory', 'collectionTexture',
       'collectedCounts', 'collectionLevel', 'threeUrl', 'fVideo', 'fAudio',
       'collectionUnit', 'clickCounts', 'isHighQuality'],
      dtype='object')


In [13]:
# 先将era, culture两列拼上去
processed_df = my_df.copy()
EXTRACTED_JSONL = DATA_DIR / "extracted" / "extracted_metadata_20260402_104557.jsonl"
extracted_df = pd.read_json(EXTRACTED_JSONL, lines=True)
# 把extracted_df的era和culture两列拼到processed_df上去，前者的source_id列需要等于后者的id列
processed_df = processed_df.merge(extracted_df[['source_id', 'era', 'culture']], how='left', left_on='id', right_on='source_id')
processed_df = processed_df.drop(columns=['source_id'])
front_columns = ['id','name','era','culture']
new_order = front_columns + processed_df.columns.difference(front_columns).tolist()
processed_df = processed_df[new_order]
processed_df.head()

,id,name,era,culture,categoryName,clickCounts,collectedCounts,collectionLevel,collectionTexture,collectionUnit,...,fVideo,image_count,imgUrl,isHighQuality,local_image_paths,museumName,pictureIds,size,threeUrl,yearName
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,陶器,3,0,2.0,陶,172,...,NaN,10.0,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,images\003895c08aa84c2a98d3d1696db7ffcf_0.JPG|...,济宁市兖州区博物馆,"2904f6a2a8aa4894a710cccf09a9cd88,4dfefd607e264...",口径19.2,NaN,新石器时代
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,陶器,0,0,4.0,陶,603,...,NaN,6.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\0043E78E1BCD4F11902DCA3C76607652_0.jpg|...,胶州市博物馆,"46f71875-7390-4f55-b09f-7abee46dd761,9b0a1818-...",口径10.5厘米,NaN,新石器时代
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,陶器,0,0,4.0,陶,114,...,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\004658C82423474EB46AA00F6BEEDAFC_0.jpg|...,桓台县博物馆,"3e177eee-85c2-4dbd-a31d-cc04d413736d,44144ea9-...",高*口径*底径：15.3*7.5*6.2,NaN,新石器时代
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,陶器,0,0,5.0,陶,93,...,NaN,6.0,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,images\0053DF2D67D346059E75B0729A69FE5B_0.jpg|...,青岛市黄岛区博物馆,"055c5f72-c8db-43c0-95dc-e9349942e728,6cc1ecc7-...",口径13，底径7，高5,NaN,新石器时代
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,陶器,0,0,4.0,陶,211,...,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,images\005C79C1C40F4AB9AA9183E5B383E158_0.jpg|...,莒南县博物馆,"1025b8b89e134b449abfac567fc59787,1b00e6180e6a4...",通高*腹围*口径*流长：39.5*37*9.1*9.9,NaN,新石器时代


In [14]:
# time列本身有的就很少，先留空
# dimensionsDesc是size
# structuredDimensions是size的结构化版本，先留空，之后统一提取
# fullDesc是description
# features是fullDesc的结构化版本，先留空，之后统一提取
# 先把这几列处理了之后再看其他列吧
processed_df.rename(columns={'size': 'dimensionsDesc', 'description': 'fullDesc'}, inplace=True)

# 现在应该将df中有但processed_df中没有的列都创建到processed_df中，然后先按照df中的列顺序排序，最后添加上那些processed_df中有但df中没有的列
# 先找出df中有但processed_df中没有的列
new_cols = df.columns.difference(processed_df.columns)
for col in new_cols:
    processed_df[col] = None
remaining_cols = processed_df.columns.difference(df.columns)
whole_order = df.columns.tolist() + remaining_cols.tolist()
processed_df = processed_df[whole_order]

In [15]:
# 先重点填一下images列和shape_type列
# shape_type列就是extracted_df中的root_shape列，直接通过id和source_id查表填写即可

# 1. 先将提取表变成一个 ID 到 Shape 的映射字典（或 Series）
shape_map = extracted_df.set_index('source_id')['root_shape']

# 2. 直接将映射结果填入原本的 shape_type 列
# 如果原本是全空，直接赋值即可；如果想只填充空值，可以用 combine_first 或 fillna
processed_df['shape_type'] = processed_df['id'].map(shape_map)
processed_df.head()

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,fAudio,fVideo,image_count,imgUrl,isHighQuality,local_image_paths,museumName,pictureIds,threeUrl,yearName
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,None,口径19.2,None,夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,None,None,...,http://www.wwsdw.net/sdimg/,NaN,10.0,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,images\003895c08aa84c2a98d3d1696db7ffcf_0.JPG|...,济宁市兖州区博物馆,"2904f6a2a8aa4894a710cccf09a9cd88,4dfefd607e264...",NaN,新石器时代
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,None,口径10.5厘米,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\0043E78E1BCD4F11902DCA3C76607652_0.jpg|...,胶州市博物馆,"46f71875-7390-4f55-b09f-7abee46dd761,9b0a1818-...",NaN,新石器时代
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,None,高*口径*底径：15.3*7.5*6.2,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\004658C82423474EB46AA00F6BEEDAFC_0.jpg|...,桓台县博物馆,"3e177eee-85c2-4dbd-a31d-cc04d413736d,44144ea9-...",NaN,新石器时代
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,None,口径13，底径7，高5,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,images\0053DF2D67D346059E75B0729A69FE5B_0.jpg|...,青岛市黄岛区博物馆,"055c5f72-c8db-43c0-95dc-e9349942e728,6cc1ecc7-...",NaN,新石器时代
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,None,通高*腹围*口径*流长：39.5*37*9.1*9.9,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,images\005C79C1C40F4AB9AA9183E5B383E158_0.jpg|...,莒南县博物馆,"1025b8b89e134b449abfac567fc59787,1b00e6180e6a4...",NaN,新石器时代


In [16]:
# images列填写需要细致
# 目标格式示例：[{'id': '3701022180000110010777-B-1', 'url': 'http://www.wwsdw.net/sdimg/picture/thumbs/d4a9ehfep/5ykd5/640x426_3701022180000110010777-B-1.jpg', 'extension': 'jpg', 'name': None, 'description': None}]
# 从json的角度，这是一个对象数组，每个对象包括id、url、extension、name、description五个字段，其中id是图片的id，url是图片的url，extension是图片的扩展名，name和description先留空。不过，它在python内存中存储的形式可能是一个字典列表。
# 当前processed_df中的local_image_paths列示例：images\003895c08aa84c2a98d3d1696db7ffcf_0.JPG|images\003895c08aa84c2a98d3d1696db7ffcf_1.JPG|images\003895c08aa84c2a98d3d1696db7ffcf_2.JPG|images\003895c08aa84c2a98d3d1696db7ffcf_3.JPG
# 这是一个用|符号分割的多个图片路径字符串
# 需要将imgUrl转换成上述格式，转换后的结果放在images列中
def transform_img_url_to_dict_list(img_url_str):
    if not img_url_str or not isinstance(img_url_str, str):
        return []
    
    # 1. 按照 '|' 分割路径
    raw_paths = img_url_str.split('|')
    
    images_list = []
    for p_str in raw_paths:
        # 使用 Path 处理反斜杠路径，Path 会自动处理平台差异
        # 即使输入是 Windows 格式的反斜杠，Path(p_str) 也能正确识别文件名
        p = Path(p_str)
        
        # 提取信息
        file_name_with_ext = p.name             # 包含后缀的文件名
        file_stem = p.stem                      # 不含后缀的文件名 (id)
        file_suffix = p.suffix.lstrip('.')      # 扩展名 (不带点)
        
        # 2. 构建符合要求的字典
        img_obj = {
            "id": file_stem,
            "url": f"images/wwsdw/{file_name_with_ext}",
            "extension": file_suffix,
            "name": None,
            "description": None
        }
        images_list.append(img_obj)
        
    return images_list

processed_df['images'] = processed_df['local_image_paths'].apply(transform_img_url_to_dict_list)
processed_df.head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,fAudio,fVideo,image_count,imgUrl,isHighQuality,local_image_paths,museumName,pictureIds,threeUrl,yearName
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,None,口径19.2,None,夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,None,None,...,http://www.wwsdw.net/sdimg/,NaN,10.0,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,images\003895c08aa84c2a98d3d1696db7ffcf_0.JPG|...,济宁市兖州区博物馆,"2904f6a2a8aa4894a710cccf09a9cd88,4dfefd607e264...",NaN,新石器时代
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,None,口径10.5厘米,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\0043E78E1BCD4F11902DCA3C76607652_0.jpg|...,胶州市博物馆,"46f71875-7390-4f55-b09f-7abee46dd761,9b0a1818-...",NaN,新石器时代
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,None,高*口径*底径：15.3*7.5*6.2,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\004658C82423474EB46AA00F6BEEDAFC_0.jpg|...,桓台县博物馆,"3e177eee-85c2-4dbd-a31d-cc04d413736d,44144ea9-...",NaN,新石器时代
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,None,口径13，底径7，高5,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,images\0053DF2D67D346059E75B0729A69FE5B_0.jpg|...,青岛市黄岛区博物馆,"055c5f72-c8db-43c0-95dc-e9349942e728,6cc1ecc7-...",NaN,新石器时代
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,None,通高*腹围*口径*流长：39.5*37*9.1*9.9,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,images\005C79C1C40F4AB9AA9183E5B383E158_0.jpg|...,莒南县博物馆,"1025b8b89e134b449abfac567fc59787,1b00e6180e6a4...",NaN,新石器时代


In [17]:
# 下面填写currentLocation列，该列格式为{'country': None, 'province': None, 'city': None, 'district': None, 'specificAddress': '枣庄市博物馆', 'coordinate': None}
# 用processed_df['museumName']来填充specificAddress字段，country统一填"中国"，province统一填"山东省"，city、district、coordinate先留空
def transform_museum_name_to_location(museum_name):
    location = {
        'country': '中国',
        'province': '山东省',
        'city': None,
        'district': None,
        'specificAddress': museum_name,
        'coordinate': None
    }
    return location
processed_df['currentLocation'] = processed_df['museumName'].apply(transform_museum_name_to_location)
processed_df.head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,fAudio,fVideo,image_count,imgUrl,isHighQuality,local_image_paths,museumName,pictureIds,threeUrl,yearName
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,None,口径19.2,None,夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,None,None,...,http://www.wwsdw.net/sdimg/,NaN,10.0,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,images\003895c08aa84c2a98d3d1696db7ffcf_0.JPG|...,济宁市兖州区博物馆,"2904f6a2a8aa4894a710cccf09a9cd88,4dfefd607e264...",NaN,新石器时代
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,None,口径10.5厘米,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\0043E78E1BCD4F11902DCA3C76607652_0.jpg|...,胶州市博物馆,"46f71875-7390-4f55-b09f-7abee46dd761,9b0a1818-...",NaN,新石器时代
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,None,高*口径*底径：15.3*7.5*6.2,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\004658C82423474EB46AA00F6BEEDAFC_0.jpg|...,桓台县博物馆,"3e177eee-85c2-4dbd-a31d-cc04d413736d,44144ea9-...",NaN,新石器时代
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,None,口径13，底径7，高5,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,images\0053DF2D67D346059E75B0729A69FE5B_0.jpg|...,青岛市黄岛区博物馆,"055c5f72-c8db-43c0-95dc-e9349942e728,6cc1ecc7-...",NaN,新石器时代
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,None,通高*腹围*口径*流长：39.5*37*9.1*9.9,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,images\005C79C1C40F4AB9AA9183E5B383E158_0.jpg|...,莒南县博物馆,"1025b8b89e134b449abfac567fc59787,1b00e6180e6a4...",NaN,新石器时代


In [18]:
# 本来就基本没法填的：time
# 要用大模型提的：structuredDimensions要从dimensionsDesc里提取（fullDesc里可能也有），features要从fullDesc里提取，excavationLocation可能会出现在fullDesc里，excavationDate可能会出现在fullDesc里
# 可以优化但暂时凑付的：currentLocation已经凑合完了；sourceCitation目前基本填不了，可以先统一填{'sourceId': 'wwsdw', 'locator': '', 'locatorType': 'Page', 'note': None}来凑付；collectionInfo可以先统一填{'collectorName': '', 'collectorId': None, 'collectedTime': {'year': None, 'month': None, 'day': None}, 'notes': None}来凑付

In [19]:
# print(df["sourceCitation"].value_counts())
# # 直接生成布尔掩码
# mask = [d.get('sourceId') != 'wwsd' if isinstance(d, dict) else False for d in df['sourceCitation']]
# filtered_df = df[mask]
# filtered_df

In [20]:
# print(df["collectionInfo"].value_counts())

In [21]:
source_dict = {'sourceId': 'wwsdw', 'locator': '', 'locatorType': 'Page', 'note': None}
processed_df["sourceCitation"] = [source_dict for _ in range(len(processed_df))]
collection_dict = {'collectorName': '', 'collectorId': None, 'collectedTime': {'year': None, 'month': None, 'day': None}, 'notes': None}
processed_df["collectionInfo"] = [collection_dict for _ in range(len(processed_df))]
time_dict = {'year': None, 'month': None, 'day': None}
processed_df["time"] = [time_dict for _ in range(len(processed_df))]
processed_df.head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,fAudio,fVideo,image_count,imgUrl,isHighQuality,local_image_paths,museumName,pictureIds,threeUrl,yearName
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.2,None,夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,None,None,...,http://www.wwsdw.net/sdimg/,NaN,10.0,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,images\003895c08aa84c2a98d3d1696db7ffcf_0.JPG|...,济宁市兖州区博物馆,"2904f6a2a8aa4894a710cccf09a9cd88,4dfefd607e264...",NaN,新石器时代
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径10.5厘米,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\0043E78E1BCD4F11902DCA3C76607652_0.jpg|...,胶州市博物馆,"46f71875-7390-4f55-b09f-7abee46dd761,9b0a1818-...",NaN,新石器时代
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：15.3*7.5*6.2,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,images\004658C82423474EB46AA00F6BEEDAFC_0.jpg|...,桓台县博物馆,"3e177eee-85c2-4dbd-a31d-cc04d413736d,44144ea9-...",NaN,新石器时代
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,"{'year': None, 'month': None, 'day': None}",口径13，底径7，高5,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,6.0,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,images\0053DF2D67D346059E75B0729A69FE5B_0.jpg|...,青岛市黄岛区博物馆,"055c5f72-c8db-43c0-95dc-e9349942e728,6cc1ecc7-...",NaN,新石器时代
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高*腹围*口径*流长：39.5*37*9.1*9.9,None,NaN,None,None,...,http://www.wwsdw.net/sdimg/,NaN,3.0,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,images\005C79C1C40F4AB9AA9183E5B383E158_0.jpg|...,莒南县博物馆,"1025b8b89e134b449abfac567fc59787,1b00e6180e6a4...",NaN,新石器时代


In [ ]:
# 下面要调用大模型统一处理剩下的四列
# structuredDimensions要从dimensionsDesc里提取（fullDesc里可能也有），features要从fullDesc里提取，excavationLocation可能会出现在fullDesc里，excavationDate可能会出现在fullDesc里
# structruedDimensions格式：假如dimensionDesc为“口径9.8厘米，腹径10.4厘米，底径6厘米，通高8.6厘米。”，则应提取为[{'unit': 'cm', 'value': 9.8, 'deviation': None, 'range': None, 'label': '口径'}, {'unit': 'cm', 'value': 10.4, 'deviation': None, 'range': None, 'label': '腹径'}, {'unit': 'cm', 'value': 6.0, 'deviation': None, 'range': None, 'label': '底径'}, {'unit': 'cm', 'value': 8.6, 'deviation': None, 'range': None, 'label': '高'}]
# features格式：下面细致观察

In [34]:
import pandas as pd

# 1. 准备基础列，避免全表操作浪费内存
base_cols = ['id', 'name', 'fullDesc']
# 确保 features 列不含空值，否则 explode 会报错
df_temp = df[base_cols + ['features']].copy()

# 后面发现似乎存在features列中不同feature的label可能会重复的情况，先检查一下这个情况的普遍程度
df_temp["feature_count"] = df_temp["features"].apply(lambda x:len(x))
df_temp["feature_set"] = df_temp["features"].apply(lambda x:set([feature.get('label') for feature in x]))
df_temp["have_duplicate_feature_labels"] = df_temp.apply(lambda row: len(row["feature_set"]) < row["feature_count"], axis=1)
# 运行上面几行观察结果，发现只有两行存在这一问题，去掉它们后再运行后续内容即可
df_temp = df_temp[~df_temp["have_duplicate_feature_labels"]].drop(columns=["feature_count", "feature_set", "have_duplicate_feature_labels"])

# 2. 将 features 列表“炸开”成多行
# 这一步后，每一行只包含一个 {'label': '...', 'value': '...'} 字典
df_exploded = df_temp.explode('features').dropna(subset=['features'])

# 3. 将字典展开为 'label' 和 'value' 两列
# pd.json_normalize 是处理这类字典列表的神器
features_flat = pd.json_normalize(df_exploded['features'])

# 4. 将展开的列拼回 ID（注意索引对齐）
df_exploded = df_exploded.reset_index(drop=True)
df_combined = pd.concat([df_exploded[base_cols], features_flat], axis=1)

# 5. 执行透视操作：将 label 列的内容转化为标题，value 填充
# index 指定保留的原始列，columns 指定哪个列的内容变表头
df_final = df_combined.pivot(index=base_cols, columns='label', values='value').reset_index()

# 6. 强反馈：查看结果
print(f"转换完成！现在的列名有: {df_final.columns.tolist()}")
df_final.head()

转换完成！现在的列名有: ['id', 'name', 'fullDesc', 'color', 'craft', 'material', 'pattern', 'shape', 'state']


label,id,name,fullDesc,color,craft,material,pattern,shape,state
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,NaN,NaN,夹砂红陶,折腹处印压出一周斜点纹,侈口，上腹斜置，折腹后收为小平底,NaN
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,暂无描述,NaN,NaN,褐陶,NaN,NaN,NaN
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,暂无描述,NaN,NaN,红陶,NaN,NaN,NaN
3,0080D7CDCAED42E99F3186AA3816DD20,新石器时代大汶口文化红褐陶盆,暂无描述,NaN,NaN,红褐陶,NaN,NaN,NaN
4,00DB2EE22C974FA0AEDE4EEE468A619D,新石器时代大汶口文化夹砂红褐陶瓶,暂无描述,NaN,NaN,夹砂红褐陶,NaN,NaN,NaN


In [42]:
df_final["color"].value_counts()

color
红色                   8
彩绘                   6
灰                    6
红                    5
红褐色                  4
                    ..
红色陶衣                 1
褐色                   1
褐陶                   1
黑陶                   1
红陶衣，白、赭色，黑色（唇饰黑彩）    1
Name: count, Length: 61, dtype: int64

In [ ]:
# color：颜色。x色，x陶，单字颜色，更复杂的描述，各种都有
# craft：工艺。镂空，镂孔，磨光等等工艺特点
# material：材质。最完整的一列，基本是夹砂/泥质+x色+陶
# pattern：纹样。对纹样的细致描述（但彩绘/镂孔/素面都混进来了）
# shape：形状。基本是辅助器型的描述，例如高柄、双耳等。
# state：状态。“残”为主，也有完整、保存较好等说法。

# 彩陶命名：年代+文化+纹饰+外形+质地+器型，纹饰+外形+质地可以对应pattern+shape+material，另外state（残）也比较常见，至于颜色color往往包含在material里，工艺craft比较复杂但也相对不重要


In [47]:
# 下面该把工作交给大模型了，先把processed_df整个存盘，另起一个notebook进行所有后续处理。
OUTPUT_JSONL = DATA_DIR / "processed_df.jsonl"
processed_df.to_json(OUTPUT_JSONL, orient="records", lines=True, force_ascii=False)